# Численная оценка максимального правдоподобия

## Задание

Необходимо составить максимально лаконичную процедуру `p_est = MLE(pdf, x, p0)`, которая численно оценивает параметры по заданной выборке. Функция `pdf(t, p)` задаёт плотность, `x` содержит выборку, `p0` содержит начальные приближения, а результат `p_est` должен содержать оценки параметров.

## Решение

In [22]:
import math
import numpy as np
from scipy.optimize import minimize

In [ ]:
def my_pdf(t, p):
    mean, std_dev = p
    exponent = -((t - mean) ** 2) / (2 * std_dev ** 2)
    coefficient = 1 / (std_dev * math.sqrt(2 * math.pi))
    return coefficient * np.exp(exponent)


def MLE(pdf, x, p0, logpdf=None):
    if logpdf is None:
        objective = lambda p: -np.log(pdf(x, p)).sum()
    else:
        objective = lambda p: -logpdf(x, p).sum()
    return minimize(objective, p0).x


In [31]:
rng = np.random.default_rng(42)
x = rng.normal(loc=5.0, scale=2.0, size=1000)

def normal_logpdf_for_mle(t, p):
    mean, log_std = p
    standardized = (t - mean) * np.exp(-log_std)
    return -0.5 * math.log(2 * math.pi) - log_std - 0.5 * standardized**2

p0 = np.array([0.0, 0.0])  # mean=0, log_std=0, so std=exp(0)=1
p_est = MLE(my_pdf, x, p0, logpdf=normal_logpdf_for_mle)
mean_est, log_std_est = p_est
std_est = np.exp(log_std_est)

print(f"Число наблюдений: {x.size}")
print(f"Начальные параметры: mean = {p0[0]}; std = {np.exp(p0[1])}")
print(f"Оценки MLE: mean = {mean_est}; std = {std_est}")


Число наблюдений: 1000
Начальные параметры: mean = 0.0; std = 1.0
Оценки MLE: mean = 4.942216958771253; std = 1.977444741432247


## Теория

Пусть $x_1,\ldots,x_n$ — наблюдения, а $f(x;p)$ — заданная плотность с вектором неизвестных параметров $p$. При независимых наблюдениях функция правдоподобия равна

$$L(p)=\prod_{i=1}^{n}f(x_i;p).$$

Оценка максимального правдоподобия выбирает значение параметров, при котором $L(p)$ максимально. Произведение плотностей неудобно для численных вычислений, поэтому используется логарифм:

$$\ell(p)=\log L(p)=\sum_{i=1}^{n}\log f(x_i;p).$$

Логарифм возрастает, поэтому максимум $L(p)$ и максимум $\ell(p)$ достигаются при одном и том же $p$. Функция `minimize` ищет минимум отрицательного логарифмического правдоподобия. В общем случае оно вычисляется как `-np.log(pdf(x, p)).sum()`, а для нормального примера ниже логарифм плотности считается напрямую через `logpdf`, чтобы избежать численного округления плотностей до нуля.